## Database

### Create DB Schema

In [1]:
import psycopg2

POSTGRES_HOST: str = "localhost"
POSTGRES_PORT: int = 5432
POSTGRES_DBNAME: str = "dms_meta"
POSTGRES_USER: str = "dms"
POSTGRES_PASSWORD: str = "dms"

pg_conn = psycopg2.connect(
    host=POSTGRES_HOST,
    port=POSTGRES_PORT,
    database=POSTGRES_DBNAME,
    user=POSTGRES_USER,
    password=POSTGRES_PASSWORD,
)

In [2]:
from pathlib import Path
current_directory = Path.cwd()

In [3]:


schema_path = current_directory / "database" / "schemas" / "schema.sql"
if not schema_path.exists():
    raise RuntimeError(f"Schema file not found at {schema_path}")

with pg_conn.cursor() as cur:
    with open(schema_path, "r", encoding="utf-8") as f:
        cur.execute(f.read())

print("Schema ensured (documents, ocr_results)")

Schema ensured (documents, ocr_results)


In [4]:
pg_conn.commit()

In [5]:
with pg_conn.cursor() as cur:
    cur.execute("""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_name = 'documents';
    """)
    columns = cur.fetchall()

for col in columns:
    print(col)


('id', 'uuid')
('file_size', 'bigint')
('created_at', 'timestamp with time zone')
('updated_at', 'timestamp with time zone')
('mime_type', 'character varying')
('document_type', 'character varying')
('linked_entity', 'character varying')
('linked_entity_id', 'character varying')
('hash_sha256', 'character varying')
('text_extraction_status', 'character varying')
('processing_status', 'character varying')
('source_filename', 'character varying')
('blob_path', 'character varying')
('acu_result_blob_path', 'character varying')


In [6]:
with pg_conn.cursor() as cur:
    cur.execute("""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_name = 'extraction_jobs';
    """)
    columns = cur.fetchall()

for col in columns:
    print(col)


('id', 'uuid')
('document_id', 'uuid')
('created_at', 'timestamp with time zone')
('started_at', 'timestamp with time zone')
('completed_at', 'timestamp with time zone')
('acu_result_blob_path', 'character varying')
('error_message', 'text')
('celery_task_id', 'character varying')
('status', 'character varying')
('analyzer_id', 'character varying')


In [3]:
from src.dms.adapters import AzureBlobStorageClient, PostgresMetadataRepository
from src.dms.service import DmsService
import os

In [4]:
from azure.storage.blob import BlobServiceClient
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

CONTAINER_NAME = "documents"

container_client = blob_service_client.get_container_client(CONTAINER_NAME)

try:
    container_client.create_container()
except Exception as e:
    print(e)
    pass  # container likely already exists

The specified container already exists.
RequestId:8697f841-c01e-00da-469b-9c6b00000000
Time:2026-02-13T03:49:17.9234405Z
ErrorCode:ContainerAlreadyExists
Content: <?xml version="1.0" encoding="utf-8"?><Error><Code>ContainerAlreadyExists</Code><Message>The specified container already exists.
RequestId:8697f841-c01e-00da-469b-9c6b00000000
Time:2026-02-13T03:49:17.9234405Z</Message></Error>


In [5]:
storage_client = AzureBlobStorageClient(blob_service_client)
metadata_repo = PostgresMetadataRepository(pg_conn)

dms_service = DmsService(storage_client=storage_client, metadata_repository=metadata_repo)

In [8]:
from datetime import datetime

# Resolve sample file
sample_pdf_path = current_directory / "data" / "AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.19_11788293_EX-10.19_Content License Agreement.pdf"
assert sample_pdf_path.exists(), "Sample PDF not found"

DOCUMENT_TYPE: str = "license-agreement"

document_id = dms_service.upload_document(
    file_path=sample_pdf_path,
    document_type=DOCUMENT_TYPE,
    source_filename=sample_pdf_path.name,
)

print("Uploaded:", document_id)

metadata = dms_service.get_document(document_id=document_id)
print("Metadata keys:", sorted(list(metadata.keys())) if metadata else None)

# List jobs
jobs = dms_service.get_extraction_jobs(document_id=document_id)
print(f"\nExtraction jobs created: {len(jobs)}")
for job in jobs:
    print(f"- Job ID: {job['id']}")
    print(f"  Status: {job['status']}")
    print(f"  Created: {job['created_at']}")



Uploaded: 526a4d34-dbf0-4f2d-b29f-0a331ac37975
Metadata keys: ['acu_result_blob_path', 'blob_path', 'created_at', 'document_type', 'file_size', 'hash_sha256', 'id', 'linked_entity', 'linked_entity_id', 'mime_type', 'processing_status', 'source_filename', 'text_extraction_status', 'updated_at']

Extraction jobs created: 1
- Job ID: a0239d76-e08e-4c4e-8215-92e004d0a331
  Status: pending
  Created: 2026-02-13 03:49:58.526553+00:00


In [9]:
downloaded = dms_service.download_document(document_id=document_id)
print("Downloaded bytes:", len(downloaded) if downloaded else None)

Downloaded bytes: 176502


In [10]:
from psycopg2.extras import RealDictCursor

with pg_conn.cursor(cursor_factory=RealDictCursor) as cur:
    cur.execute("""
        SELECT *
        FROM documents
        ORDER BY created_at DESC;
    """)
    rows = cur.fetchall()

for row in rows:
    print(row)
    print("/n")


RealDictRow({'id': '526a4d34-dbf0-4f2d-b29f-0a331ac37975', 'source_filename': 'AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.19_11788293_EX-10.19_Content License Agreement.pdf', 'blob_path': 'raw/license-agreement/526a4d34-dbf0-4f2d-b29f-0a331ac37975.pdf', 'file_size': 176502, 'mime_type': 'application/pdf', 'document_type': 'license-agreement', 'linked_entity': None, 'linked_entity_id': None, 'hash_sha256': '2c5fe2d4f89cd8f305e99eefac75177858c8fd57155b4347a3b010e270e3ae8a', 'text_extraction_status': 'ready', 'processing_status': 'pending extraction', 'acu_result_blob_path': None, 'created_at': datetime.datetime(2026, 2, 13, 3, 49, 58, 512203, tzinfo=datetime.timezone.utc), 'updated_at': datetime.datetime(2026, 2, 13, 3, 49, 58, 512203, tzinfo=datetime.timezone.utc)})
/n
RealDictRow({'id': 'adbfcd80-14b0-4c40-b9db-c4c45f5257ac', 'source_filename': 'AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.19_11788293_EX-10.19_Content License Agreement.pdf', 'blob_path': 'raw/license-agreemen

In [11]:
print("=== Simulating processing ===\n")

# 1) Mark ready
print("1. Marking 'ready'...")
dms_service.update_textextraction_status(
    document_id=document_id,
    status="ready",
)
print("   ✓ text_extraction_status → 'ready'")

# 2) ACU running
print("\n2. ACU running...")
dms_service.mark_acu_running(
    document_id=document_id,
)
print("   ✓ processing_status → 'acu running'")

# 3) Completing processing
print("\n3. Completing processing...")
dms_service.update_textextraction_status(
    document_id=document_id,
    status="completed",
)
dms_service.mark_processing_done(
    document_id=document_id,
)
print("   ✓ text_extraction_status → 'completed'")
print("   ✓ processing_status → 'done'")

# 4) Mark latest extraction job as done (if exists)
jobs = dms_service.get_extraction_jobs(document_id=document_id)
if jobs:
    # pick most recent job if your SQL orders DESC; otherwise this is still safe as "a job"
    job_id = jobs[0]["id"]
    dms_service.update_extraction_job(
        job_id=job_id,
        status="done",
        error_message=None,
    )
    print(f"   ✓ extraction_jobs[{job_id}] → 'done'")
else:
    print("   (no extraction jobs found for this document)")


=== Simulating processing ===

1. Marking 'ready'...
   ✓ text_extraction_status → 'ready'

2. ACU running...
   ✓ processing_status → 'acu running'

3. Completing processing...
   ✓ text_extraction_status → 'completed'
   ✓ processing_status → 'done'
   ✓ extraction_jobs[a0239d76-e08e-4c4e-8215-92e004d0a331] → 'done'


In [12]:
# Final document status
document = dms_service.get_document(document_id=document_id)
print("=== Final Document ===")
print(f"ID: {document_id}")
print(f"Filename: {document.get('source_filename')}")
print(f"Text extraction status: {document.get('textextraction_status', 'N/A')}")
print(f"Processing status: {document.get('processing_status', 'N/A')}")

# Final job status
jobs = dms_service.get_extraction_jobs(document_id=document_id)
print("\n=== Extraction Jobs ===")
for job in jobs:
    print(f"Job ID: {job['id']}")
    print(f"Status: {job['status']}")
    print(f"Created: {job['created_at']}")
    print(f"Completed: {job.get('completed_at', 'N/A')}")
    print(f"Error: {job.get('error_message', 'None')}")

=== Final Document ===
ID: 526a4d34-dbf0-4f2d-b29f-0a331ac37975
Filename: AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.19_11788293_EX-10.19_Content License Agreement.pdf
Text extraction status: N/A
Processing status: done

=== Extraction Jobs ===
Job ID: a0239d76-e08e-4c4e-8215-92e004d0a331
Status: done
Created: 2026-02-13 03:49:58.526553+00:00
Completed: 2026-02-13 03:55:44.421226+00:00
Error: None
